In [ ]:
import pandas as pd
import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
import seaborn as sns 
from scipy.stats import pearsonr
import json
from shapely.geometry import shape 
import json 
from pandas.tseries.offsets import Week
from statsmodels.tsa.stattools import kpss 
import statsmodels.api as sm
import warnings
import scipy.stats as stats
from scipy.stats import zscore

### Filter Dataframes 

In [ ]:
def filter_columns(df):

    # columns_to_keep = ['timestamp', 'precipitation (mm)', 'temperature (degrees Celsius)', 'windgusts (m/s)', 'windspeed (m/s)']
    columns_to_keep = ['timestamp', 'precipitation (mm)', 'temperature (degrees Celsius)']

    
    # Filter the DataFrame to keep only the specified columns
    filtered_df = df[columns_to_keep].copy()

    filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
        
    # Define a dictionary for renaming columns
    rename_mapping = {
        'timestamp': 'Timestamp',
        'precipitation (mm)': 'Precipitation (mm)',
        'temperature (degrees Celsius)': 'Temperature (°C)',
        # 'windgusts (m/s)': 'Wind Gusts (m/s)',
        # 'windspeed (m/s)': 'Wind Speed (m/s)'
    }

    # Rename the columns using the mapping
    filtered_df = filtered_df.rename(columns=rename_mapping)

    return filtered_df

In [ ]:
def filter_columns_rev2(df):

    # columns_to_keep = ['timestamp', 'precipitation (mm)', 'temperature (degrees Celsius)', 'windgusts (m/s)', 'windspeed (m/s)']
    columns_to_keep = ['timestamp', 'precipitation (mm)', 'temperature (degrees Celsius)']
    
    # Filter the DataFrame to keep only the specified columns
    filtered_df = df[columns_to_keep].copy()

    filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
    
    # Define a dictionary for renaming columns
    rename_mapping = {
        'timestamp': 'Timestamp',
        'lightningdistance (km)': 'Lightning Distance', 
        'lightningevents (-)': 'Lightning Events',
        'precipitation (mm)': 'Precipitation (mm)',
        'temperature (degrees Celsius)': 'Temperature (°C)',
        # 'windgusts (m/s)': 'Wind Gusts (m/s)',
        # 'windspeed (m/s)': 'Wind Speed (m/s)'
    }

    # Rename the columns using the mapping
    filtered_df = filtered_df.rename(columns=rename_mapping)
    
    return filtered_df

In [ ]:
def filter_columns_rev3(df):

    # columns_to_keep = ['timestamp', 'precipitation S001265 (mm)', 'temperature (degrees Celsius)', 'windgusts (m/s)', 'windspeed (m/s)']
    columns_to_keep = ['timestamp', 'precipitation S001265 (mm)', 'temperature (degrees Celsius)']
    
    # Filter the DataFrame to keep only the specified columns
    filtered_df = df[columns_to_keep].copy()

    filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])

    # Define a dictionary for renaming columns
    rename_mapping = {
        'timestamp': 'Timestamp',
        'lightningdistance (km)': 'Lightning Distance', 
        'lightningevents (-)': 'Lightning Events',
        'precipitation S001265 (mm)': 'Precipitation (mm)',
        'temperature (degrees Celsius)': 'Temperature (°C)',
        # 'windgusts (m/s)': 'Wind Gusts (m/s)',
        # 'windspeed (m/s)': 'Wind Speed (m/s)'
    }
    
    # Rename the columns using the mapping
    filtered_df = filtered_df.rename(columns=rename_mapping)

    return filtered_df

### 5 min re-index 

In [ ]:
def fill_missing_timestamps_to_new_list_v2(df_list, timestamp_col='Timestamp', freq='5min'):
    updated_dfs = []

    for i, df in enumerate(df_list):
        if timestamp_col in df.columns:
            df = df.copy()  # Avoid SettingWithCopyWarning

            # Convert to datetime and set index
            df[timestamp_col] = pd.to_datetime(df[timestamp_col])
            df.set_index(timestamp_col, inplace=True)

            # Generate complete timestamp range
            full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)
            original_len = len(df)
            df_reindexed = df.reindex(full_range)
            df_reindexed.index.name = timestamp_col

            # Print NaN summary
            total_entries = df_reindexed.shape[0] * df_reindexed.shape[1]
            total_nans = df_reindexed.isna().sum().sum()
            nan_percent = (total_nans / total_entries) * 100 if total_entries > 0 else 0

            print(f"\nDataFrame {i+1}:")
            print(f"- Original length: {original_len}")
            print(f"- After reindexing: {len(df_reindexed)} rows")
            print(f"- Total NaNs: {total_nans:,} ({nan_percent:.2f}%)")

            updated_dfs.append(df_reindexed.reset_index())
    
    return updated_dfs

### Linear Interpolation 

In [ ]:
def interpolate_cleaned_df_linear_v2(df, col='Temperature (°C)'):
    df_interp = df.copy()

    # Set datetime index
    if 'Timestamp' in df_interp.columns:
        df_interp = df_interp.set_index('Timestamp')

    if not isinstance(df_interp.index, pd.DatetimeIndex):
        raise ValueError("DataFrame index must be a DatetimeIndex")

    # New column for interpolated values
    new_col = f'Interpolated {col}'
    df_interp[new_col] = df_interp[col]

    # Compute % of NaNs per day based on the original column
    nan_percent = (
        df_interp[col]
        .isna()
        .groupby(df_interp.index.normalize())
        .mean() * 100
    )

    # Create the NaN_Percent_Per_Day column
    df_interp['NaN_Percent_Per_Day'] = df_interp.index.normalize().map(nan_percent)

    # Group by day and interpolate
    grouped = df_interp.groupby(df_interp.index.normalize())

    interpolated_groups = []
    for _, group in grouped:
        group[new_col] = group[new_col].interpolate(method='linear', limit_direction='both')
        interpolated_groups.append(group)

    return pd.concat(interpolated_groups)


### Remove days with high % of NaNs (Clean by Missing Threshold)

In [ ]:
def clean_5min_df_by_nan_threshold_v2(df, interp_col='Interpolated Temperature (°C)', nan_col='NaN_Percent_Per_Day', threshold_percent=10):
    df = df.copy()

    # Ensure datetime index
    if not isinstance(df.index, pd.DatetimeIndex):
        if 'Timestamp' in df.columns:
            df['Timestamp'] = pd.to_datetime(df['Timestamp'])
            df.set_index('Timestamp', inplace=True)

    # Group by day
    grouped = df.groupby(df.index.normalize())

    df['Day_Removed'] = False  # initialize
    total_bad_days = 0

    print(f"Original DataFrame length: {len(df)} rows\n")

    for day, group in grouped:
        day_nan_percent = group[nan_col].iloc[0]

        if day_nan_percent >= threshold_percent:
            df.loc[group.index, 'Day_Removed'] = True
            total_bad_days += 1

    # Filter out the bad days
    cleaned_df = df[~df['Day_Removed']].copy()

    print(f"Days removed: {total_bad_days}")
    print(f"Cleaned DataFrame length: {len(cleaned_df)} rows\n")
    print("Total NaNs after cleaning:")
    print(f"- {interp_col}: {cleaned_df[interp_col].isna().sum()}")
    print('')
    print('-------------')

    return cleaned_df

### Hourly resample & summary 

In [ ]:
def resample_hourly_and_nan_percent_full_summary(df, 
                                                 raw_col='Temperature (°C)', 
                                                 interp_col='Interpolated Temperature (°C)'):
    df = df.copy()

    # Ensure datetime index
    if not isinstance(df.index, pd.DatetimeIndex):
        if 'Timestamp' in df.columns:
            df['Timestamp'] = pd.to_datetime(df['Timestamp'])
            df.set_index('Timestamp', inplace=True)

    # Round to hourly for grouping
    df['Hour'] = df.index.floor('h')

    # Get full hourly range from min to max date
    full_hour_range = pd.date_range(df['Hour'].min(), df['Hour'].max(), freq='H')
    full_hour_df = pd.DataFrame(index=full_hour_range)
    full_hour_df.index.name = 'Hour'

    # Resample using mean (for temperature)
    interpolated_hourly = df.groupby('Hour')[interp_col].mean().to_frame()
    full_hour_df = full_hour_df.join(interpolated_hourly)

    # Determine if each hour is NaN in raw data
    raw_hour_nan_flags = df.groupby('Hour')[raw_col].apply(lambda x: x.isna().all())
    full_hour_df['Raw_NaN_Hour'] = full_hour_df.index.map(raw_hour_nan_flags).fillna(True).astype(int)

    # Add month info
    full_hour_df['YearMonth'] = full_hour_df.index.to_period('M')

    # Summary table: % NaN hours per month
    summary = full_hour_df.groupby('YearMonth').agg(
        Total_Hours=('Raw_NaN_Hour', 'count'),
        NaN_Hours=('Raw_NaN_Hour', 'sum')
    )
    summary['Percent_NaN_Hours'] = (summary['NaN_Hours'] / summary['Total_Hours']) * 100
    summary = summary.reset_index()
    summary['YearMonth'] = summary['YearMonth'].astype(str)

    print("\n Hourly NaN Summary Per Month:")
    print("----------------------------------")
    for _, row in summary.iterrows():
        ym = row['YearMonth']
        pct = row['Percent_NaN_Hours']
        print(f" - {ym}: {pct:.2f}% NaN hours")

    return full_hour_df, summary


### Function to remove months (10% thresh) & create month summary 

In [ ]:
def drop_months_with_missing_days(df, threshold_percent=10, location='Legon', variable='Temp'):
    df = df.copy()

    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame index must be a DatetimeIndex")

    df = df.sort_index()

    # Full daily range
    full_date_range = pd.date_range(df.index.min().normalize(), df.index.max().normalize(), freq='D')
    full_dates_df = pd.DataFrame(index=full_date_range)
    full_dates_df['HasData'] = False
    full_dates_df.loc[df.index.normalize(), 'HasData'] = True
    full_dates_df['YearMonth'] = full_dates_df.index.to_period('M')

    summary_records = []
    valid_months = []

    print("\n Missing Days Summary Per Month:")
    print("----------------------------------")

    for ym, group in full_dates_df.groupby('YearMonth'):
        total_days = len(group)
        present_days = group['HasData'].sum()
        missing_days = total_days - present_days
        missing_percent = (missing_days / total_days) * 100

        # Determine status (only Valid or Removed)
        status = 'Valid' if missing_percent <= threshold_percent else 'Removed'

        summary_records.append({
            'YearMonth': ym,
            'Total_Days': total_days,
            'Present_Days': present_days,
            'Missing_Days': missing_days,
            'Missing_Percent': missing_percent,
            'Status': status,
            'Location': location,
            'Variable': variable
        })

        print(f" - {ym}: {missing_percent:.2f}% missing ({missing_days} days) → {status}")

        if status == 'Valid':
            valid_months.append(ym)

    # Filter DataFrame
    df['Month_Period'] = df.index.to_period('M')
    df_filtered = df[df['Month_Period'].isin(valid_months)].copy()
    df_filtered.drop(columns='Month_Period', inplace=True)

    print(f"\n Months retained (valid): {len(valid_months)}")
    print(" - " + ", ".join(str(m) for m in valid_months))

    summary_df = pd.DataFrame(summary_records)
    summary_df['YearMonth'] = summary_df['YearMonth'].astype(str)

    return df_filtered, summary_df


### Read weather station data  

In [ ]:
tahmo_16 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00016.csv')
tahmo_98 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00098.csv')

## temasco station 
tahmo_118 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00118.csv')

tahmo_126 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00126.csv')
tahmo_127 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00127.csv')
tahmo_313 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00313.csv')
tahmo_319 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00319.csv')
tahmo_391 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00391.csv')
tahmo_567 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00567.csv')
tahmo_647 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00647.csv')
tahmo_651 = pd.read_csv('/Users/kwamedonkor/Downloads/UW/Research/TAHMO/With_Lightning_Distance/TAHMO_export_673b722af8aada2ee55cef6b/TA00651.csv')

### --- Workflow for one weather station (Temasco) shown --- 

### Process and resample data 

In [ ]:
# filter station data  
temasco_df = filter_columns(tahmo_118)

temasco_df_r1 = temasco_df[['Timestamp', 'Temperature (°C)']]

## 5 minute reindex 
temasco_df_re_indexed = fill_missing_timestamps_to_new_list_v2([temasco_df_r1], timestamp_col='Timestamp', freq='5min')

## linear interpolation 
temasco_interpol = interpolate_cleaned_df_linear_v2(temasco_df_re_indexed[0])

### remove dates with lots of missing values (in this case, 90%)
temasco_cleaned = clean_5min_df_by_nan_threshold_v2(temasco_interpol, threshold_percent=90)

## hourly resampling 
full_hourly_resampled, hourly_summary = resample_hourly_and_nan_percent_full_summary(temasco_cleaned)

### Merge Hour & Month Summary Stats 

In [ ]:
## monthly resampling --> to drop months with more than 10% missing days 
full_month_df, monthly_summary = drop_months_with_missing_days(temasco_cleaned, threshold_percent=10, location='Temasco')

merged_stats = pd.merge(hourly_summary, monthly_summary, on = 'YearMonth')

# merged_stats.to_excel('/Users/kwamedonkor/Downloads/UW/Research/Quals/Output_Files/Temp/merged_stats_temasco.xlsx')

### Final Hourly Resampled df --> to Merge with other dfs    

In [ ]:
temasco_final_hr_resampled_v2 = full_hourly_resampled.copy()

temasco_final_hr_resampled_v2 = temasco_final_hr_resampled_v2[['Interpolated Temperature (°C)']]
temasco_final_hr_resampled_v2 = temasco_final_hr_resampled_v2.rename(columns = {'Interpolated Temperature (°C)':'Interpol_Temp_Temasco'})

# temasco_final_hr_resampled_v2.to_excel('/Users/kwamedonkor/Downloads/UW/Research/Quals/Output_Files/Temp/temasco_final_hr_resampled_v2.xlsx')